In [1]:
import numpy as np
import pandas as pd

# Load Excel file
file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Distributed Generation/Datasets/dg9bus.xlsx'
excel_file = pd.ExcelFile(file_path)

# Load each sheet into dataframes
transmission_df = pd.read_excel(excel_file, sheet_name='BUS 9 (Transmisi)')
gen_load_df = pd.read_excel(excel_file, sheet_name='BUS 9 (Generator Load)')
trafo_df = pd.read_excel(excel_file, sheet_name='BUS 9 (Trafo)')

# Preprocess Data
bus_data = gen_load_df[['bus', 'kapasitas_generator_MVA', 'kapasitas_load_MW', 'kapasitas_load_MVAR']]
transmission_data = transmission_df[['line', 'from_bus_trans', 'to_bus_trans', 'length_km', 'R0_ohm_km', 'R1_ohm_km', 'induktansi_H_km', 'kapasitansi_C_km']]
trafo_data = trafo_df[['trafo', 'from_bus_trafo', 'to_bus_trafo', 'trafo_MVA']]

# Manually input cap_trans_MVA
cap_trans_MVA = {
    1: 98.3,
    2: 63.1,
    3: 217,
    4: 150,
    5: 217.2,
    6: 45.4
}

# Function to calculate losses
def calculate_losses_all_sheets(transmission_data, trafo_data, cap_trans_MVA, dg_bus=None, dg_capacity=0):
    R0 = transmission_data['R0_ohm_km'].values
    R1 = transmission_data['R1_ohm_km'].values
    length = transmission_data['length_km'].values
    L = transmission_data['induktansi_H_km'].values
    C = transmission_data['kapasitansi_C_km'].values
    f = 50  # Frequency
    X_L = 2 * np.pi * f * L
    X_C = 1 / (2 * np.pi * f * C)
    Z0 = np.sqrt(R0**2 + (X_L - X_C)**2)
    Z1 = np.sqrt(R1**2 + (X_L - X_C)**2)
    losses_line = np.sum(Z0 * length) + np.sum(Z1 * length)
    trafo_losses = np.sum(trafo_data['trafo_MVA'].values) * 0.01
    
    if dg_bus is not None:
        dg_indices = (transmission_data['from_bus_trans'] == dg_bus) | (transmission_data['to_bus_trans'] == dg_bus)
        losses_after_dg = (
            np.sum(Z0[~dg_indices] * length[~dg_indices]) +
            np.sum(Z1[~dg_indices] * length[~dg_indices]) +
            np.sum(Z0[dg_indices] * length[dg_indices] * (1 - dg_capacity)) +
            np.sum(Z1[dg_indices] * length[dg_indices] * (1 - dg_capacity))
        )
        trafo_bus_indices = (trafo_data['from_bus_trafo'] == dg_bus) | (trafo_data['to_bus_trafo'] == dg_bus)
        trafo_losses_after_dg = (
            np.sum(trafo_data['trafo_MVA'][~trafo_bus_indices].values * 0.01) +
            np.sum(trafo_data['trafo_MVA'][trafo_bus_indices].values * 0.01 * (1 - dg_capacity))
        )
    else:
        losses_after_dg = losses_line
        trafo_losses_after_dg = trafo_losses

    total_losses_before = losses_line + trafo_losses
    total_losses_after = losses_after_dg + trafo_losses_after_dg

    return total_losses_before, total_losses_after

# Fitness function
def fitness_function_with_dg_all_sheets(position, transmission_data, trafo_data, cap_trans_MVA):
    dg_bus = int(position[0])
    dg_capacity = position[1]
    _, losses_after = calculate_losses_all_sheets(transmission_data, trafo_data, cap_trans_MVA, dg_bus, dg_capacity)
    return losses_after

# Update bats function
def update_bats(bats, best_bat_position, f_min, f_max, Q_min, Q_max, A_min, A_max, r_min, r_max, min_position, max_position):
    num_bats, dim = bats.shape
    updated_bats = bats.copy()
    
    for i in range(num_bats):
        f_i = f_min + (f_max - f_min) * np.random.rand()
        Q_i = Q_min + (Q_max - Q_min) * np.random.rand()
        A_i = A_min + (A_max - A_min) * np.random.rand()
        r_i = r_min + (r_max - r_min) * np.random.rand()

        velocity = f_i * (best_bat_position - bats[i])
        position_update = A_i * (np.random.rand(dim) * Q_i - bats[i])
        new_position = bats[i] + velocity + position_update

        for d in range(dim):
            new_position[d] = max(min_position[d], min(max_position[d], new_position[d]))

        if np.random.rand() > r_i:
            new_position += np.random.uniform(-0.1, 0.1, size=dim)
        
        updated_bats[i] = new_position

    return updated_bats

# Bat Algorithm for optimization
def bat_algorithm_dg(transmission_data, trafo_data, cap_trans_MVA, num_bats=10, num_iterations=100):
    dim = 2  # Two parameters: [bus_index, DG_capacity]
    bats = np.zeros((num_bats, dim))
    bats[:, 0] = np.random.randint(0, bus_data.shape[0], size=num_bats)
    bats[:, 1] = np.random.uniform(0, 1, size=num_bats)

    f_min, f_max = 0, 2
    Q_min, Q_max = 0, 1
    A_min, A_max = 0.5, 1.0
    r_min, r_max = 0.5, 1.0
    min_position = [0, 0]
    max_position = [bus_data.shape[0] - 1, 0.5]

    fitness = np.array([fitness_function_with_dg_all_sheets(bat, transmission_data, trafo_data, cap_trans_MVA) for bat in bats])
    best_idx = np.argmin(fitness)
    best_position = bats[best_idx].copy()
    best_fitness = fitness[best_idx]

    for _ in range(num_iterations):
        bats = update_bats(bats, best_position, f_min, f_max, Q_min, Q_max, A_min, A_max, r_min, r_max, min_position, max_position)
        fitness = np.array([fitness_function_with_dg_all_sheets(bat, transmission_data, trafo_data, cap_trans_MVA) for bat in bats])

        for i in range(num_bats):
            if fitness[i] < best_fitness:
                best_fitness = fitness[i]
                best_position = bats[i].copy()

    return best_position, best_fitness

# Calculate pre-optimization losses
pre_optimization_losses, _ = calculate_losses_all_sheets(transmission_data, trafo_data, cap_trans_MVA)
pre_optimization_losses_mw = pre_optimization_losses / 1_000_000

# Run Bat Algorithm
best_dg_position, post_optimization_losses = bat_algorithm_dg(transmission_data, trafo_data, cap_trans_MVA)
post_optimization_losses_mw = post_optimization_losses / 1_000_000

max_dg_capacity_kw = 500
optimal_dg_capacity_kW = best_dg_position[1] * max_dg_capacity_kw

# Output results
print(f"Pre-optimization losses (MW): {pre_optimization_losses_mw}")
print(f"Optimal DG placement at bus: {int(best_dg_position[0])}")
print(f"Optimal DG capacity in kW: {optimal_dg_capacity_kW}")
print(f"Post-optimization losses (MW): {post_optimization_losses_mw}")

Pre-optimization losses (MW): 488.0739443100552
Optimal DG placement at bus: 6
Optimal DG capacity in kW: 299.7589731998227
Post-optimization losses (MW): 373.6517483569884
